# DSFB Computer Graphics Reviewer Notebook

This notebook is a Google Colab entry point for the `dsfb-computer-graphics` crate. It exists to make the crate easy to evaluate from a clean browser session without relying on a preconfigured local workstation or an already-built Rust environment. The notebook orchestrates the Rust crate rather than replacing it: Rust remains the source of truth for scene generation, Demo A, Demo B, metrics, figures, reports, and artifact layout.

In this crate, DSFB is presented as a supervisory trust layer for temporal reuse. Demo A is the primary bounded synthetic experiment. It uses a deterministic scene with a moving occluder, a disocclusion event, thin-geometry worst cases, a fixed-alpha TAA baseline, a stronger residual-threshold baseline, and a DSFB path that changes only the supervisory control logic. The point of Demo A is to make behavioral differences legible: stale history persists in the baseline, DSFB lowers trust in the exposed region, and the blend weight moves toward the current frame earlier.

Demo B is a bounded fixed-budget adaptive-sampling study that reuses the trust field from Demo A. It does not claim to be a full temporal SAR controller. Instead, it shows how the same supervisory signal can be used to redistribute a fixed budget toward the reveal ROI in a controlled synthetic setting.

The notebook produces timestamped outputs under `output-dsfb-computer-graphics/` so each run is preserved rather than overwritten. That structure matters for reproducibility and technical review: a reviewer can archive one run directory, inspect the exact generated figures and reports later, and compare multiple runs without accidental loss of prior outputs.

The notebook also generates a PDF reviewer bundle and a ZIP archive automatically. The PDF is useful when a reviewer wants one compact document containing the overview, the key metrics, the figures, and the limitations. The ZIP is useful when a reviewer or transition partner wants the complete artifact set from a single run directory, including machine-readable metrics and the raw generated figures.

What to expect from this notebook:

- It installs only the dependencies needed to run the crate and the crate-local bundle script.
- It clones the repository, builds the crate, creates a timestamped run directory, runs the crate end-to-end, generates the PDF and ZIP, and displays the major artifacts inline.
- It shows where the files were written so you can inspect or download them directly.

What this notebook does not claim:

- It does not claim optimal TAA or optimal adaptive sampling.
- It does not claim production integration completeness.
- It does not claim measured production GPU timings.
- It does not claim field readiness, funding outcomes, or licensing outcomes.

The experiment is intentionally bounded and synthetic, but it is organized so a reviewer can make a rapid technical judgment from real generated outputs rather than from qualitative prose alone.

“The experiment is intended to demonstrate behavioral differences rather than establish optimal performance.”

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, Markdown, SVG, display

REPO_URL = "https://github.com/infinityabundance/dsfb.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/dsfb")
CRATE_DIR = REPO_DIR / "crates" / "dsfb-computer-graphics"
NOTEBOOK_PATH = CRATE_DIR / "colab" / "dsfb_computer_graphics_demo.ipynb"
BUNDLE_SCRIPT = CRATE_DIR / "colab" / "build_artifact_bundle.py"
OUTPUT_ROOT = CRATE_DIR / "output-dsfb-computer-graphics"

print(f"Repository URL: {REPO_URL}")
print(f"Repository branch: {REPO_BRANCH}")
print(f"Notebook path inside repo: {NOTEBOOK_PATH.relative_to(REPO_DIR) if REPO_DIR.exists() else 'crates/dsfb-computer-graphics/colab/dsfb_computer_graphics_demo.ipynb'}")

## Environment Setup

The next cell installs the small set of system/runtime dependencies the notebook needs, installs Rust if it is missing, clones the repository if it is not already present in the Colab runtime, verifies `cargo` and `rustc`, and builds the crate. The system package `librsvg2-bin` is installed so the crate-local bundling script can convert SVG figures into raster images for the PDF bundle.

In [ ]:
def run(command: str, cwd: Path | None = None) -> None:
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True, cwd=str(cwd) if cwd else None, executable="/bin/bash")

run("apt-get update")
run("apt-get install -y librsvg2-bin zip")
run("python -m pip install --quiet pillow ipywidgets pandas")

if shutil.which("cargo") is None:
    run("curl https://sh.rustup.rs -sSf | sh -s -- -y")

cargo_bin = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"

if not REPO_DIR.exists():
    run(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    print(f"Reusing existing checkout at {REPO_DIR}")

run("rustc --version")
run("cargo --version")
run("cargo build --all-features", cwd=CRATE_DIR)

os.chdir(CRATE_DIR)
print(f"Working directory set to {Path.cwd()}")

## Run Configuration

This cell creates the crate-local output root `output-dsfb-computer-graphics/` and a timestamped run directory inside it. The timestamped naming convention is deliberate: it prevents overwrite-by-default behavior and makes it easier to archive multiple runs for review.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
run_name = f"output-dsfb-computer-graphics-{timestamp}"
run_dir = OUTPUT_ROOT / run_name
suffix = 1
while run_dir.exists():
    run_name = f"output-dsfb-computer-graphics-{timestamp}-r{suffix:02d}"
    run_dir = OUTPUT_ROOT / run_name
    suffix += 1

run_dir.mkdir(parents=True, exist_ok=False)
print(f"Output root: {OUTPUT_ROOT}")
print(f"Run directory: {run_dir}")

## Demo Execution

The notebook uses the crate's unified `run-all` command to generate Demo A and Demo B into the same timestamped run directory. The Rust side writes the figures, reports, metrics, debug fields, and an artifact manifest. The crate-local bundle script then builds the PDF and ZIP from those real outputs.

In [ ]:
run(f"cargo run -- run-all --output '{run_dir}'", cwd=CRATE_DIR)
run(f"python '{BUNDLE_SCRIPT}' --run-dir '{run_dir}'", cwd=CRATE_DIR)

## Artifact Discovery

The crate writes an `artifact_manifest.json` file into the run directory. The notebook uses that manifest to locate the generated files instead of relying on brittle hard-coded guesses.

In [ ]:
manifest = json.loads((run_dir / "artifact_manifest.json").read_text())
metrics = json.loads((run_dir / manifest["demo_a"]["metrics_path"]).read_text())
summary = metrics["summary"]
canonical = next((scenario for scenario in metrics["scenarios"] if scenario["scenario_id"] == "thin_reveal"), metrics["scenarios"][0])

def find_run(scenario: dict, run_id: str) -> dict:
    return next(run["summary"] for run in scenario["runs"] if run["summary"]["run_id"] == run_id)

fixed = find_run(canonical, "fixed_alpha")
strong = find_run(canonical, "strong_heuristic")
host = find_run(canonical, "dsfb_host_realistic")

demo_b_metrics = json.loads((run_dir / manifest["demo_b"]["metrics_path"]).read_text())
demo_b_canonical = next((scenario for scenario in demo_b_metrics["scenarios"] if scenario["scenario_id"] == "thin_reveal"), demo_b_metrics["scenarios"][0])

def find_policy(scenario: dict, policy_id: str) -> dict:
    return next(policy for policy in scenario["policies"] if policy["policy_id"] == policy_id)

uniform_policy = find_policy(demo_b_canonical, "uniform")
combined_policy = find_policy(demo_b_canonical, "combined_heuristic")
imported_policy = find_policy(demo_b_canonical, "imported_trust")
pdf_path = run_dir / manifest["pdf_bundle_file_name"]
zip_path = run_dir.parent / manifest["zip_bundle_file_name"]

display(Markdown(f"**Run directory:** `{run_dir}`"))
display(Markdown(f"**PDF bundle:** `{pdf_path}`"))
display(Markdown(f"**ZIP bundle:** `{zip_path}`"))
display(Markdown(f"**Primary result:** {summary['primary_behavioral_result']}"))
if summary.get("secondary_behavioral_result"):
    display(Markdown(f"**Secondary result:** {summary['secondary_behavioral_result']}"))
display(Markdown(f"**Scenario count:** {len(metrics['scenarios'])} Demo A scenarios, {len(demo_b_metrics['scenarios'])} Demo B scenarios."))

In [ ]:
import pandas as pd

summary_table = pd.DataFrame(
    [
        {
            "Metric": "Ghost persistence frames",
            "Fixed-alpha": fixed["ghost_persistence_frames"],
            "Strong heuristic": strong["ghost_persistence_frames"],
            "Host-realistic DSFB": host["ghost_persistence_frames"],
        },
        {
            "Metric": "Peak ROI error",
            "Fixed-alpha": round(fixed["peak_roi_mae"], 5),
            "Strong heuristic": round(strong["peak_roi_mae"], 5),
            "Host-realistic DSFB": round(host["peak_roi_mae"], 5),
        },
        {
            "Metric": "Cumulative ROI error",
            "Fixed-alpha": round(fixed["cumulative_roi_mae"], 5),
            "Strong heuristic": round(strong["cumulative_roi_mae"], 5),
            "Host-realistic DSFB": round(host["cumulative_roi_mae"], 5),
        },
        {
            "Metric": "Average non-ROI MAE",
            "Fixed-alpha": round(fixed["average_non_roi_mae"], 5),
            "Strong heuristic": round(strong["average_non_roi_mae"], 5),
            "Host-realistic DSFB": round(host["average_non_roi_mae"], 5),
        },
    ]
)

display(Markdown("## Demo A Canonical Metrics"))
display(summary_table)

display(Markdown(
    f"Canonical onset frame: **{canonical['onset_frame']}**, host onset-response latency: **{host['onset_response_latency_frames']}**, "
    f"strong-heuristic onset-response latency: **{strong['onset_response_latency_frames']}**, "
    f"host trust/error rank correlation: **{host['trust_error_rank_correlation']:.4f}**."
))
display(Markdown(f"Mixed or neutral Demo A scenarios surfaced explicitly: **{', '.join(summary['mixed_or_neutral_scenarios'])}**"))

demo_b_table = pd.DataFrame(
    [
        {
            "Metric": "ROI MAE",
            "Uniform": round(uniform_policy["roi_mae"], 5),
            "Combined heuristic": round(combined_policy["roi_mae"], 5),
            "Imported trust": round(imported_policy["roi_mae"], 5),
        },
        {
            "Metric": "ROI RMSE",
            "Uniform": round(uniform_policy["roi_rmse"], 5),
            "Combined heuristic": round(combined_policy["roi_rmse"], 5),
            "Imported trust": round(imported_policy["roi_rmse"], 5),
        },
        {
            "Metric": "ROI mean spp",
            "Uniform": round(uniform_policy["roi_mean_spp"], 2),
            "Combined heuristic": round(combined_policy["roi_mean_spp"], 2),
            "Imported trust": round(imported_policy["roi_mean_spp"], 2),
        },
    ]
)
display(Markdown("## Demo B Sampling Summary"))
display(demo_b_table)
display(Markdown(f"Demo B mixed or neutral scenarios: **{', '.join(demo_b_metrics['summary']['neutral_or_mixed_scenarios'])}**"))

## Inline Artifact Display

The next cell renders the major figures directly in the notebook so a reviewer can inspect the system diagram, the trust map, the before/after comparison, the trust-vs-error plot, and the Demo B sampling figure without downloading anything first.

In [ ]:
figure_titles = {
    "fig_system_diagram.svg": "Figure 1. System Diagram",
    "fig_trust_map.svg": "Figure 2. Trust Map",
    "fig_before_after.svg": "Figure 3. Before / After",
    "fig_trust_vs_error.svg": "Figure 4. Trust vs Error",
    "fig_intervention_alpha.svg": "Figure 5. Intervention and Alpha",
    "fig_ablation.svg": "Figure 6. Ablation Comparison",
    "fig_roi_nonroi_error.svg": "Figure 7. ROI vs Non-ROI Error",
    "fig_leaderboard.svg": "Figure 8. Aggregate Leaderboard",
    "fig_scenario_mosaic.svg": "Figure 9. Scenario Mosaic",
}
for relative_path in manifest["demo_a"]["figure_paths"]:
    title = figure_titles.get(Path(relative_path).name, Path(relative_path).name)
    display(Markdown(f"### {title}"))
    display(SVG(filename=str(run_dir / relative_path)))

for relative_path in manifest["demo_b"]["figure_paths"]:
    display(Markdown(f"### {Path(relative_path).name}"))
    display(SVG(filename=str(run_dir / relative_path)))

In [ ]:
reviewer_summary = (run_dir / manifest["demo_a"]["reviewer_summary_path"]).read_text()
report_excerpt = "\n".join((run_dir / manifest["demo_a"]["report_path"]).read_text().splitlines()[:120])
reviewer_reports = manifest.get("reviewer_report_paths", [])

display(Markdown("## Reviewer Summary"))
display(Markdown(reviewer_summary))
if reviewer_reports:
    display(Markdown("## Additional Reviewer Reports"))
    for relative_path in reviewer_reports:
        display(Markdown(f"- `{relative_path}`"))
display(Markdown("## Report Excerpt"))
display(Markdown("```markdown\n" + report_excerpt + "\n```"))

## Download Controls

Use the buttons below to download the PDF reviewer bundle or the ZIP archive for the current run directory.

In [ ]:
import ipywidgets as widgets
from google.colab import files

pdf_button = widgets.Button(
    description="Download PDF",
    button_style="success",
    layout=widgets.Layout(width="220px")
)
zip_button = widgets.Button(
    description="Download ZIP",
    button_style="primary",
    layout=widgets.Layout(width="220px")
)

def download_pdf(_):
    files.download(str(pdf_path))

def download_zip(_):
    files.download(str(zip_path))

pdf_button.on_click(download_pdf)
zip_button.on_click(download_zip)

display(HTML("<b>Current run bundle:</b>"))
display(widgets.HBox([pdf_button, zip_button]))